# Aether Stage 2 — full frozen-Qwen R1 baseline

This run scales the successful R1 smoke to the complete `train-clean-100 + train-clean-360` cache exposed by `karl4th/limmim-v2-en`.

- Stage 1 AetherSpeech: frozen
- Qwen3-4B: frozen
- R1 Connector: initialized from the best R1 smoke WER checkpoint, then trainable
- optimizer and scheduler: new
- hard ceiling: 100,000 optimizer steps
- evaluation: every 1,000 steps on 256 fixed validation examples
- automatic plateau stop: WER improvement below 0.5 absolute percentage points for 3 consecutive eligible evaluations
- adaptive microbatch: 16 on 96 GB, 8 on 80 GB, 4 on 40 GB (effective batch always 16)
- all run artifacts: unique `MyDrive/aether-v3/stage2/runYYMMDD-HHMMSS/`

Each operational task has its own cell.

## 0. Mount Google Drive

In [ ]:
import json, logging, os, shutil, subprocess, sys, time
from pathlib import Path
from google.colab import drive

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", force=True)
drive.mount("/content/drive")
print("DRIVE: mounted", flush=True)

## 1. Checkout the exact `stage2` branch and install

In [ ]:
REPO_URL = "https://github.com/karl4th/aether-v3.git"
REPO_DIR = "/content/aether-v3"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", "stage2"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", "stage2", "--single-branch", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "stage2"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/stage2"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR, "huggingface_hub"], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(Path(REPO_DIR)/"src"))
for module_name in list(sys.modules):
    if module_name == "aether_v3" or module_name.startswith("aether_v3."):
        del sys.modules[module_name]
print("CODE:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(), flush=True)

## 2. Read the private Hugging Face token

In [ ]:
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
assert os.environ["HF_TOKEN"], "Add HF_TOKEN in Colab Secrets and enable notebook access"
print("HF TOKEN: available (value hidden)", flush=True)

## 3. Load the full-run config and create a unique Drive run

In [ ]:
import torch
from aether_v3.config import load_config
from aether_v3.training.stage2_utils import create_run_dir

cfg = load_config("configs/stage2_r1_full.yaml")
assert cfg.llm.model_id == "Qwen/Qwen3-4B"
assert not cfg.connector.resampler.enabled and cfg.connector.resampler.ratio == 1
assert cfg.stage2_train.max_steps == 100000
assert cfg.stage2_train.eval_interval == 1000 and cfg.stage2_train.eval_steps == []
assert cfg.stage2_train.eval_max_examples == 256
assert cfg.stage2_train.plateau_metric == "wer"
assert cfg.stage2_train.plateau_min_delta == 0.005
assert cfg.stage2_train.plateau_patience_evals == 3
assert torch.cuda.is_available(), "Select a GPU runtime"
assert torch.cuda.is_bf16_supported(), "This full run requires a BF16-capable GPU"

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
if vram_gb >= 90:
    cfg.stage2_train.batch_size, cfg.stage2_train.grad_accum_steps = 16, 1
    gpu_profile = "96GB RTX PRO 6000 profile"
elif vram_gb >= 70:
    cfg.stage2_train.batch_size, cfg.stage2_train.grad_accum_steps = 8, 2
    gpu_profile = "80GB H100 profile"
else:
    cfg.stage2_train.batch_size, cfg.stage2_train.grad_accum_steps = 4, 4
    gpu_profile = "40GB A100 profile"
assert cfg.stage2_train.batch_size * cfg.stage2_train.grad_accum_steps == 16

run_dir = create_run_dir(cfg.stage2_train.drive_root)
cache_drive_root = Path(cfg.stage2_data.cache_dir)
cache_local_root = Path("/content/aether-stage2-r1-full-v2-en-cache")
print("GPU:", gpu_name, f"({vram_gb:.1f} GB)", flush=True)
print("GPU PROFILE:", gpu_profile, flush=True)
print("RUN:", run_dir, flush=True)
print(
    "TRAIN CONFIG:",
    f"microbatch={cfg.stage2_train.batch_size}",
    f"grad_accum={cfg.stage2_train.grad_accum_steps}",
    "effective_batch=16",
    flush=True,
)


## 4. Find and validate the best R1 smoke checkpoint

In [ ]:
candidates = []
for path in Path(cfg.stage2_train.drive_root).glob("run*/best_wer.pt"):
    try:
        checkpoint = torch.load(path, map_location="cpu", weights_only=False)
        source_cfg = checkpoint.get("config", {})
        source_train = source_cfg.get("stage2_train", {})
        source_llm = source_cfg.get("llm", {})
        source_resampler = source_cfg.get("connector", {}).get("resampler", {})
        if (
            source_llm.get("model_id") == "Qwen/Qwen3-4B"
            and source_train.get("max_steps") == 5000
            and source_train.get("batch_size") == 2
            and source_train.get("grad_accum_steps") == 8
            and source_resampler.get("enabled") is False
            and source_resampler.get("ratio") == 1
        ):
            candidates.append((path.stat().st_mtime, path, checkpoint.get("step"), checkpoint.get("metrics", {})))
    except Exception as exc:
        print("SKIP:", path, repr(exc), flush=True)
assert candidates, "No matching R1 smoke best_wer.pt found under MyDrive/aether-v3/stage2/run*"
_, source_checkpoint, source_step, source_metrics = max(candidates)
cfg.stage2_train.init_trainable_from = str(source_checkpoint)
print("SOURCE CHECKPOINT:", source_checkpoint, flush=True)
print("SOURCE STEP:", source_step, flush=True)
print("SOURCE METRICS:", json.dumps(source_metrics, indent=2), flush=True)

## 5. Download and load the private Stage 1 checkpoint

In [ ]:
from huggingface_hub import hf_hub_download
from aether_v3.models.aether_speech import AetherSpeechEncoder
from aether_v3.training.stage2_utils import load_stage1_encoder

stage1_path = hf_hub_download(
    repo_id=cfg.stage2_train.stage1_repo_id,
    filename=cfg.stage2_train.stage1_filename,
    revision=cfg.stage2_train.stage1_revision,
    token=os.environ["HF_TOKEN"],
)
encoder = AetherSpeechEncoder(cfg.aether_speech)
stage1_checkpoint = load_stage1_encoder(stage1_path, encoder)
encoder.eval()
print("STAGE 1:", stage1_path, "step=", stage1_checkpoint.get("step", "unknown"), flush=True)

## 6. Load Qwen3-4B and initialize only the Connector weights

In [ ]:
from transformers import AutoTokenizer
from aether_v3.models.aether_speech_llm import AetherSpeechLLM
from aether_v3.training.train_stage2 import load_stage2_trainable_weights

tokenizer = AutoTokenizer.from_pretrained(cfg.llm.model_id, revision=cfg.llm.revision)
print("Loading Qwen3-4B...", flush=True)
model = AetherSpeechLLM(cfg.aether_speech, cfg.connector, cfg.llm, speech_frozen=True).cuda()
load_stage1_encoder(stage1_path, model.encoder)
model.connector.to(dtype=next(model.llm.parameters()).dtype)
source_loaded = load_stage2_trainable_weights(model, source_checkpoint, "cuda")
assert model.connector.bridge.output_scale.dtype == torch.float32
assert not any(p.requires_grad for p in model.llm.parameters())
assert not any(p.requires_grad for p in model.encoder.parameters())
print("CONNECTOR: loaded weights-only from step", source_loaded.get("step"), flush=True)
print("OUTPUT SCALE:", float(model.connector.bridge.output_scale.detach()), flush=True)
print("TRAINABLE PARAMETERS:", sum(p.numel() for p in model.parameters() if p.requires_grad), flush=True)

## 7. Open the complete, disjoint train and validation streams

In [ ]:
from datasets import load_dataset

train_rows = load_dataset(
    cfg.stage2_data.dataset_id,
    split=cfg.stage2_data.train_split,
    token=os.environ["HF_TOKEN"],
)
validation_rows = load_dataset(
    cfg.stage2_data.dataset_id,
    split=cfg.stage2_data.validation_split,
    token=os.environ["HF_TOKEN"],
)
assert len(train_rows) == 132553, f"limmim-v2-en train count mismatch: {len(train_rows)}"
assert len(validation_rows) == 2703, f"limmim-v2-en validation count mismatch: {len(validation_rows)}"
assert "transcript" in train_rows.column_names
assert "semantic_codes" in train_rows.column_names
assert "byte_target" in train_rows.column_names
print("DATA: limmim-v2-en verified", {"train": len(train_rows), "validation": len(validation_rows)}, flush=True)


## 8. Build or resume the complete persistent training cache

In [ ]:
from aether_v3.data.stage2_cache import build_limmim_stage2_shards

print("TRAIN CACHE: building/resuming all 132553 utterances", flush=True)
train_count = build_limmim_stage2_shards(
    train_rows, model.encoder, tokenizer, cache_drive_root/"train", "train",
    encode_batch_size=32, shard_size=512, max_examples=None,
    progress_total=len(train_rows), show_progress=True,
)
assert train_count == 132553, f"Training cache count mismatch: {train_count}"
print("TRAIN CACHE COMPLETE:", train_count, flush=True)


## 9. Build or resume the complete persistent validation cache

In [ ]:
print("VALIDATION CACHE: building/resuming all 2703 separate validation utterances", flush=True)
validation_count = build_limmim_stage2_shards(
    validation_rows, model.encoder, tokenizer, cache_drive_root/"validation", "validation",
    encode_batch_size=32, shard_size=512, max_examples=None,
    progress_total=len(validation_rows), show_progress=True,
)
assert validation_count == 2703, f"Validation cache count mismatch: {validation_count}"
dataset_manifest = {
    "dataset_id": cfg.stage2_data.dataset_id,
    "dataset_config": cfg.stage2_data.dataset_config,
    "train_split": cfg.stage2_data.train_split,
    "validation_split": cfg.stage2_data.validation_split,
    "train_examples": train_count,
    "validation_examples": validation_count,
    "persistent_cache": str(cache_drive_root),
}
(run_dir/"dataset_manifest.json").write_text(json.dumps(dataset_manifest, indent=2))
print("VALIDATION CACHE COMPLETE:", validation_count, flush=True)
print(json.dumps(dataset_manifest, indent=2), flush=True)


## 10. Stage the persistent cache onto the Colab SSD

In [ ]:
cache_local_root.mkdir(parents=True, exist_ok=True)
for split in ("train", "validation"):
    source = cache_drive_root/split
    destination = cache_local_root/split
    destination.mkdir(parents=True, exist_ok=True)
    print(f"LOCAL CACHE: syncing {split} to Colab SSD", flush=True)
    subprocess.run(["rsync", "-ah", "--info=progress2", f"{source}/", f"{destination}/"], check=True)
assert len(list((cache_local_root/"train").glob("shard-*.pt"))) > 200
assert len(list((cache_local_root/"validation").glob("shard-*.pt"))) > 1
print("LOCAL CACHE: ready", cache_local_root, flush=True)

## 11. Run a real initialized forward before the long run

In [ ]:
import gc
from torch.utils.data import DataLoader
from aether_v3.data.stage2_collate import collate_stage2_batch
from aether_v3.data.stage2_dataset import Stage2ShardDataset

diag_loader = DataLoader(
    Stage2ShardDataset(cache_local_root/"train", shuffle=False),
    batch_size=2,
    collate_fn=collate_stage2_batch,
)
raw = next(iter(diag_loader))
batch = {key: value.cuda() if torch.is_tensor(value) else value for key, value in raw.items()}
model.train()
with torch.inference_mode():
    output = model.forward_cached(batch)
assert output.loss is not None and torch.isfinite(output.loss)
assert torch.isfinite(output.logits).all()
print("PREFLIGHT FORWARD: PASS loss=", float(output.loss), flush=True)
del output, batch, raw, diag_loader
gc.collect()
torch.cuda.empty_cache()
print(
    "PREFLIGHT CLEANUP:",
    f"allocated={torch.cuda.memory_allocated()/2**30:.2f} GB",
    f"reserved={torch.cuda.memory_reserved()/2**30:.2f} GB",
    flush=True,
)


## 12. Start the full frozen-Qwen R1 run

The loop logs remaining steps, percentage and ETA every 10 optimizer steps. It evaluates every 2,500 steps. Plateau produces `plateau_report.json` and `plateau_stop.pt`; scale failure produces `abort_output_scale.pt`. All best and periodic checkpoints are written directly to Drive.

In [ ]:
from aether_v3.training.train_stage2 import run_stage2_training

torch.cuda.reset_peak_memory_stats()
print("TRAIN: starting; hard ceiling 100000 steps, automatic WER plateau stop enabled", flush=True)
print("TRAIN: eval every 1000 on 256 examples; plateau delta=0.005, patience=3", flush=True)
print(
    "TRAIN:",
    f"microbatch={cfg.stage2_train.batch_size}",
    f"grad_accum={cfg.stage2_train.grad_accum_steps}",
    "effective_batch=16",
    flush=True,
)
run_stage2_training(
    cfg,
    run_dir,
    cache_local_root/"train",
    cache_local_root/"validation",
    model=model,
    tokenizer=tokenizer,
)
for name in ("last.pt", "best_val_loss.pt", "best_wer.pt", "best_cer.pt", "training_summary.json", "provenance.json"):
    assert (run_dir/name).exists(), f"Missing artifact: {name}"
shutil.copy2(run_dir/"last.pt", run_dir/"r1_full_final.pt")
print("TRAIN FINISHED:", (run_dir/"training_summary.json").read_text(), flush=True)


## 13. Display the stopping decision and best metrics

In [ ]:
summary = json.loads((run_dir/"training_summary.json").read_text())
records = [json.loads(line) for line in (run_dir/"log.jsonl").read_text().splitlines()]
evals = [record for record in records if "val_loss" in record]
print("SUMMARY", json.dumps(summary, indent=2), flush=True)
print("EVALUATIONS", json.dumps([
    {key: record[key] for key in ("step", "val_loss", "wer", "cer")}
    for record in evals
], indent=2), flush=True)
if (run_dir/"plateau_report.json").exists():
    print("PLATEAU REPORT", (run_dir/"plateau_report.json").read_text(), flush=True)
print("BEST WER CHECKPOINT:", run_dir/"best_wer.pt", flush=True)
print("FINAL CHECKPOINT:", run_dir/"r1_full_final.pt", flush=True)